<a href="https://colab.research.google.com/github/acorrea79/techchallenge-fase2-pipeline-alfabetizacao/blob/main/notebooks/pipeline_alfabetizacao_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TechChallenge Fase 2 - Pipeline Híbrido para Análise da Alfabetização no Brasil

Projeto desenvolvido para a FIAP, turma 1IAST.

Integrante:

- Andre Correa Luis Vilas Boas

## Objetivo do notebook

Este notebook centraliza a execução acadêmica da pipeline de dados para análise da alfabetização no Brasil.

A solução utiliza:

- GCP BigQuery Sandbox;
- Google Colab;
- Python;
- Pandas;
- arquitetura Medalhão com camadas Bronze, Silver e Gold;
- ingestão Batch;
- simulação de Streaming;
- validações de qualidade;
- logs de monitoramento.

Este notebook faz parte da entrega do TechChallenge Fase 2.

## Instalação das bibliotecas

In [1]:
!pip install -q pandas numpy pyarrow google-cloud-bigquery db-dtypes

## Imports básicos do projeto

In [2]:
import os
import json
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np

from google.colab import auth
from google.cloud import bigquery

## Configurações do projeto

In [3]:
# Configurações principais do projeto

PROJECT_ID = "fiap-techchallenge-fase2"
DATASET_ID = "basedosdados.br_inep_avaliacao_alfabetizacao"

BASE_PATH = Path("/content/techchallenge-fase2-pipeline-alfabetizacao")

BRONZE_PATH = BASE_PATH / "data" / "bronze"
BRONZE_BATCH_PATH = BRONZE_PATH / "batch"
BRONZE_STREAMING_PATH = BRONZE_PATH / "streaming"

SILVER_PATH = BASE_PATH / "data" / "silver"
GOLD_PATH = BASE_PATH / "data" / "gold"
LOGS_PATH = BASE_PATH / "logs"

TABLES = {
    "alunos": f"{DATASET_ID}.alunos",
    "dicionario": f"{DATASET_ID}.dicionario",
    "meta_alfabetizacao_brasil": f"{DATASET_ID}.meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf": f"{DATASET_ID}.meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio": f"{DATASET_ID}.meta_alfabetizacao_municipio",
    "municipio": f"{DATASET_ID}.municipio",
    "uf": f"{DATASET_ID}.uf",
}

print("Configuração carregada com sucesso.")
print(f"Projeto GCP: {PROJECT_ID}")
print(f"Dataset principal: {DATASET_ID}")

Configuração carregada com sucesso.
Projeto GCP: fiap-techchallenge-fase2
Dataset principal: basedosdados.br_inep_avaliacao_alfabetizacao


## Criação das pastas da pipeline

In [4]:
# Criação da estrutura de pastas no ambiente Colab

paths = [
    BRONZE_BATCH_PATH,
    BRONZE_STREAMING_PATH,
    SILVER_PATH,
    GOLD_PATH,
    LOGS_PATH,
]

for path in paths:
    path.mkdir(parents=True, exist_ok=True)

print("Estrutura de pastas criada no Colab:")
for path in paths:
    print(f"- {path}")

Estrutura de pastas criada no Colab:
- /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch
- /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/streaming
- /content/techchallenge-fase2-pipeline-alfabetizacao/data/silver
- /content/techchallenge-fase2-pipeline-alfabetizacao/data/gold
- /content/techchallenge-fase2-pipeline-alfabetizacao/logs


## Função simples de log

In [5]:
# Função de log para monitoramento simples da pipeline

def write_log(message: str, level: str = "INFO"):
    LOGS_PATH.mkdir(parents=True, exist_ok=True)

    log_file = LOGS_PATH / "pipeline_execution.log"
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    log_line = f"{timestamp} | {level} | {message}"

    with open(log_file, "a", encoding="utf-8") as file:
        file.write(log_line + "\n")

    print(log_line)


write_log("Notebook principal iniciado.")

2026-07-01 02:24:51 | INFO | Notebook principal iniciado.


## Autenticação no Google

In [6]:
# Autenticação no Google
# Será aberta uma janela para autorizar o acesso com sua conta Google.

auth.authenticate_user()

write_log("Autenticação no Google realizada com sucesso.")

2026-07-01 02:24:51 | INFO | Autenticação no Google realizada com sucesso.


## Criação do cliente BigQuery

In [7]:
# Criação do cliente BigQuery

client = bigquery.Client(project=PROJECT_ID)

write_log("Cliente BigQuery criado com sucesso.")

2026-07-01 02:24:51 | INFO | Cliente BigQuery criado com sucesso.


##Teste de acesso ao BigQuery

In [8]:
# Teste simples de acesso ao BigQuery Sandbox

query = """
SELECT
  'BigQuery Sandbox acessado com sucesso' AS status
"""

df_test = client.query(query).to_dataframe(create_bqstorage_client=False)

display(df_test)

write_log("Consulta de teste executada com sucesso.")

,status
0,BigQuery Sandbox acessado com sucesso


2026-07-01 02:24:52 | INFO | Consulta de teste executada com sucesso.


# Listar tabelas do dataset principal

In [9]:
!gcloud auth list
!gcloud config set project fiap-techchallenge-fase2
!gcloud config get-value project

query_tables = """
SELECT
  table_catalog,
  table_schema,
  table_name,
  table_type
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA.TABLES`
ORDER BY
  table_name
"""

df_tables = client.query(query_tables).to_dataframe(create_bqstorage_client=False)

display(df_tables)

write_log(f"Listagem de tabelas executada. Total de tabelas encontradas: {len(df_tables)}")

      Credentialed Accounts
ACTIVE  ACCOUNT
*       gabrielhcorrea27@gmail.com

To set the active account, run:
    $ gcloud config set account `ACCOUNT`

Updated property [core/project].
fiap-techchallenge-fase2


,table_catalog,table_schema,table_name,table_type
0,basedosdados,br_inep_avaliacao_alfabetizacao,alunos,BASE TABLE
1,basedosdados,br_inep_avaliacao_alfabetizacao,dicionario,BASE TABLE
2,basedosdados,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_brasil,BASE TABLE
3,basedosdados,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_municipio,BASE TABLE
4,basedosdados,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_uf,BASE TABLE
5,basedosdados,br_inep_avaliacao_alfabetizacao,municipio,BASE TABLE
6,basedosdados,br_inep_avaliacao_alfabetizacao,uf,BASE TABLE


2026-07-01 02:25:01 | INFO | Listagem de tabelas executada. Total de tabelas encontradas: 7


## Listar colunas do dataset principal

In [10]:
# Listagem das colunas das tabelas do dataset principal

query_columns = """
SELECT
  table_name,
  column_name,
  data_type,
  is_nullable,
  ordinal_position
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA.COLUMNS`
ORDER BY
  table_name,
  ordinal_position
"""

df_columns = client.query(query_columns).to_dataframe(create_bqstorage_client=False)

display(df_columns)

write_log(f"Schema carregado com sucesso. Total de colunas encontradas: {len(df_columns)}")

,table_name,column_name,data_type,is_nullable,ordinal_position
0,alunos,ano,INT64,YES,1
1,alunos,id_municipio,STRING,YES,2
2,alunos,id_escola,STRING,YES,3
3,alunos,id_aluno,STRING,YES,4
4,alunos,caderno,STRING,YES,5
...,...,...,...,...,...
78,uf,proporcao_aluno_nivel_4,FLOAT64,YES,11
79,uf,proporcao_aluno_nivel_5,FLOAT64,YES,12
80,uf,proporcao_aluno_nivel_6,FLOAT64,YES,13
81,uf,proporcao_aluno_nivel_7,FLOAT64,YES,14


2026-07-01 02:25:02 | INFO | Schema carregado com sucesso. Total de colunas encontradas: 83


## Volume das tabelas

In [11]:
# Consulta de volume das tabelas
# Esta etapa apoia decisões de FinOps.
# Tentamos primeiro INFORMATION_SCHEMA.TABLE_STORAGE.
# Caso falhe por restrição de localização/metadados, usamos __TABLES__ como alternativa.

query_storage_information_schema = """
SELECT
  table_name,
  total_rows,
  ROUND(total_logical_bytes / 1024 / 1024, 2) AS tamanho_mb
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA.TABLE_STORAGE`
ORDER BY
  total_logical_bytes DESC
"""

query_storage_legacy = """
SELECT
  table_id AS table_name,
  row_count AS total_rows,
  ROUND(size_bytes / 1024 / 1024, 2) AS tamanho_mb
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.__TABLES__`
ORDER BY
  size_bytes DESC
"""

try:
    df_storage = client.query(query_storage_information_schema).to_dataframe(
        create_bqstorage_client=False
    )
    storage_source = "INFORMATION_SCHEMA.TABLE_STORAGE"
    write_log("Consulta de volume executada via INFORMATION_SCHEMA.TABLE_STORAGE.")

except Exception as error:
    write_log(
        f"Consulta via INFORMATION_SCHEMA.TABLE_STORAGE falhou. "
        f"Usando alternativa __TABLES__. Erro: {str(error)[:200]}",
        level="WARNING"
    )

    df_storage = client.query(query_storage_legacy).to_dataframe(
        create_bqstorage_client=False
    )
    storage_source = "__TABLES__"
    write_log("Consulta alternativa de volume executada via __TABLES__.")

display(df_storage)

print(f"Fonte usada para volume das tabelas: {storage_source}")

2026-07-01 02:25:02 | WARNING | Consulta via INFORMATION_SCHEMA.TABLE_STORAGE falhou. Usando alternativa __TABLES__. Erro: 404 Not found: Dataset basedosdados:br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA was not found in location US; reason: notFound, message: Not found: Dataset basedosdados:br_inep_avaliacao_alfabe
2026-07-01 02:25:03 | INFO | Consulta alternativa de volume executada via __TABLES__.


,table_name,total_rows,tamanho_mb
0,alunos,3867999,256.10
1,municipio,23995,1.75
2,meta_alfabetizacao_municipio,10704,1.10
3,uf,145,0.01
4,meta_alfabetizacao_uf,81,0.01
5,dicionario,27,0.00
6,meta_alfabetizacao_brasil,3,0.00


Fonte usada para volume das tabelas: __TABLES__


## Validação das tabelas obrigatórias

In [12]:
# Validação das tabelas obrigatórias do desafio

expected_tables = {
    "alunos",
    "dicionario",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "municipio",
    "uf",
}

found_tables = set(df_tables["table_name"].tolist())

missing_tables = expected_tables - found_tables
extra_tables = found_tables - expected_tables

validation_result = {
    "expected_tables": sorted(list(expected_tables)),
    "found_tables": sorted(list(found_tables)),
    "missing_tables": sorted(list(missing_tables)),
    "extra_tables": sorted(list(extra_tables)),
    "status": "approved" if not missing_tables else "failed"
}

display(pd.DataFrame([validation_result]))

if missing_tables:
    write_log(f"Tabelas obrigatórias ausentes: {missing_tables}", level="ERROR")
else:
    write_log("Todas as tabelas obrigatórias foram encontradas.")

,expected_tables,found_tables,missing_tables,extra_tables,status
0,"[alunos, dicionario, meta_alfabetizacao_brasil...","[alunos, dicionario, meta_alfabetizacao_brasil...",[],[],approved


2026-07-01 02:25:03 | INFO | Todas as tabelas obrigatórias foram encontradas.


## Salvar metadados no ambiente do Colab

In [13]:
# Salvando metadados de descoberta em arquivos locais do Colab

metadata_path = BRONZE_BATCH_PATH / "metadata"
metadata_path.mkdir(parents=True, exist_ok=True)

df_tables.to_csv(metadata_path / "dataset_tables.csv", index=False)
df_columns.to_csv(metadata_path / "dataset_columns.csv", index=False)
df_storage.to_csv(metadata_path / "dataset_storage.csv", index=False)

with open(metadata_path / "table_validation.json", "w", encoding="utf-8") as file:
    json.dump(validation_result, file, ensure_ascii=False, indent=4)

write_log("Metadados do dataset salvos na camada Bronze/metadata.")

2026-07-01 02:25:03 | INFO | Metadados do dataset salvos na camada Bronze/metadata.


## Verificar arquivos gerados

In [14]:
# Verificação dos arquivos gerados

for root, dirs, files in os.walk(BASE_PATH):
    level = root.replace(str(BASE_PATH), "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

    subindent = " " * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

techchallenge-fase2-pipeline-alfabetizacao/
  data/
    bronze/
      batch/
        batch_ingestion_manifest.json
        uf.csv
        meta_alfabetizacao_uf.csv
        meta_alfabetizacao_municipio.csv
        alunos_agregado.csv
        alunos_sample.csv
        dicionario.csv
        municipio.csv
        meta_alfabetizacao_brasil.csv
        metadata/
          dataset_columns.csv
          table_validation.json
          dataset_tables.csv
          dataset_storage.csv
      streaming/
    silver/
    gold/
  logs/
    pipeline_execution.log


## Nesta etapa, realizamos a ingestão batch dos dados públicos da Base dos Dados, utilizando o BigQuery Sandbox como componente cloud.

### A camada Bronze recebe os dados em formato bruto ou minimamente controlado, preservando a origem e permitindo rastreabilidade.

Estratégia adotada:

- ingestão integral das tabelas pequenas;
- ingestão controlada da tabela `alunos`, devido ao maior volume;
- geração de uma amostra de alunos;
- geração de uma visão agregada de alunos por ano, município, rede e série;
- registro de manifesto da ingestão;
- registro de logs de execução.

Essa abordagem segue a restrição acadêmica de custo zero e evita consumo desnecessário de processamento.

## Configuração da ingestão Batch

In [15]:
# Configuração da ingestão Batch

BATCH_OUTPUT_PATH = BRONZE_BATCH_PATH

SMALL_TABLES = [
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "municipio",
    "uf",
    "dicionario",
]

ALUNOS_SAMPLE_SIZE = 100000

batch_manifest = []

write_log(f"Tabelas pequenas configuradas para ingestão integral: {SMALL_TABLES}")
write_log(f"Tamanho da amostra da tabela alunos: {ALUNOS_SAMPLE_SIZE}")

2026-07-01 02:25:03 | INFO | Tabelas pequenas configuradas para ingestão integral: ['meta_alfabetizacao_brasil', 'meta_alfabetizacao_uf', 'meta_alfabetizacao_municipio', 'municipio', 'uf', 'dicionario']
2026-07-01 02:25:03 | INFO | Tamanho da amostra da tabela alunos: 100000


## Função auxiliar para executar query e salvar CSV

In [16]:
# Função auxiliar para executar consultas BigQuery e salvar resultado em CSV

def run_query_to_csv(query: str, output_file: Path, description: str):
    """
    Executa uma consulta BigQuery, converte o resultado para DataFrame
    e salva em CSV na camada Bronze.
    """

    start_time = time.perf_counter()
    started_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    write_log(f"Iniciando extração: {description}")

    try:
        job = client.query(query)
        df = job.to_dataframe(create_bqstorage_client=False)

        output_file.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_file, index=False, encoding="utf-8")

        elapsed_seconds = round(time.perf_counter() - start_time, 2)
        finished_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        file_size_mb = round(output_file.stat().st_size / 1024 / 1024, 4)

        manifest_item = {
            "description": description,
            "output_file": str(output_file),
            "rows": int(len(df)),
            "columns": int(len(df.columns)),
            "bytes_processed": int(job.total_bytes_processed or 0),
            "file_size_mb": file_size_mb,
            "started_at": started_at,
            "finished_at": finished_at,
            "elapsed_seconds": elapsed_seconds,
            "status": "success"
        }

        batch_manifest.append(manifest_item)

        write_log(
            f"Extração concluída: {description} | "
            f"linhas={len(df)} | colunas={len(df.columns)} | "
            f"arquivo={output_file.name} | tamanho_mb={file_size_mb}"
        )

        return df

    except Exception as error:
        elapsed_seconds = round(time.perf_counter() - start_time, 2)
        finished_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        manifest_item = {
            "description": description,
            "output_file": str(output_file),
            "rows": 0,
            "columns": 0,
            "bytes_processed": 0,
            "file_size_mb": 0,
            "started_at": started_at,
            "finished_at": finished_at,
            "elapsed_seconds": elapsed_seconds,
            "status": "failed",
            "error": str(error)
        }

        batch_manifest.append(manifest_item)

        write_log(f"Erro na extração: {description} | erro={error}", level="ERROR")
        raise

## Ingestão integral das tabelas pequenas

In [17]:
# Ingestão integral das tabelas pequenas para a camada Bronze

bronze_dataframes = {}

for table_name in SMALL_TABLES:
    table_id = TABLES[table_name]

    query = f"""
    SELECT *
    FROM `{table_id}`
    """

    output_file = BATCH_OUTPUT_PATH / f"{table_name}.csv"

    df = run_query_to_csv(
        query=query,
        output_file=output_file,
        description=f"Ingestão integral da tabela {table_name}"
    )

    bronze_dataframes[table_name] = df

write_log("Ingestão integral das tabelas pequenas concluída.")

2026-07-01 02:25:03 | INFO | Iniciando extração: Ingestão integral da tabela meta_alfabetizacao_brasil
2026-07-01 02:25:05 | INFO | Extração concluída: Ingestão integral da tabela meta_alfabetizacao_brasil | linhas=3 | colunas=11 | arquivo=meta_alfabetizacao_brasil.csv | tamanho_mb=0.0004
2026-07-01 02:25:05 | INFO | Iniciando extração: Ingestão integral da tabela meta_alfabetizacao_uf
2026-07-01 02:25:06 | INFO | Extração concluída: Ingestão integral da tabela meta_alfabetizacao_uf | linhas=81 | colunas=12 | arquivo=meta_alfabetizacao_uf.csv | tamanho_mb=0.005
2026-07-01 02:25:06 | INFO | Iniciando extração: Ingestão integral da tabela meta_alfabetizacao_municipio
2026-07-01 02:25:08 | INFO | Extração concluída: Ingestão integral da tabela meta_alfabetizacao_municipio | linhas=10704 | colunas=13 | arquivo=meta_alfabetizacao_municipio.csv | tamanho_mb=0.7712
2026-07-01 02:25:08 | INFO | Iniciando extração: Ingestão integral da tabela municipio
2026-07-01 02:25:12 | INFO | Extração conc

## Ingestão controlada da tabela alunos: amostra

In [18]:
# Ingestão controlada da tabela alunos
# A tabela alunos possui maior volume. Para custo zero, usamos amostra controlada.

query_alunos_sample = f"""
SELECT
  ano,
  id_municipio,
  id_escola,
  id_aluno,
  caderno,
  serie,
  rede,
  presenca,
  preenchimento_caderno,
  alfabetizado,
  proficiencia,
  peso_aluno
FROM
  `{TABLES["alunos"]}`
LIMIT {ALUNOS_SAMPLE_SIZE}
"""

alunos_sample_file = BATCH_OUTPUT_PATH / "alunos_sample.csv"

df_alunos_sample = run_query_to_csv(
    query=query_alunos_sample,
    output_file=alunos_sample_file,
    description=f"Ingestão controlada da tabela alunos com LIMIT {ALUNOS_SAMPLE_SIZE}"
)

display(df_alunos_sample.head())

write_log("Amostra da tabela alunos gerada com sucesso.")

2026-07-01 02:25:14 | INFO | Iniciando extração: Ingestão controlada da tabela alunos com LIMIT 100000
2026-07-01 02:25:25 | INFO | Extração concluída: Ingestão controlada da tabela alunos com LIMIT 100000 | linhas=100000 | colunas=12 | arquivo=alunos_sample.csv | tamanho_mb=5.3711


,ano,id_municipio,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno
0,2023,1100080,60000258,11020747,1,2,3,0,0,0,NaN,NaN
1,2023,1302405,60001114,13015211,1,2,3,0,0,0,NaN,NaN
2,2023,1506807,60001515,15088589,1,2,3,0,0,0,NaN,NaN
3,2023,1508001,60002261,15096612,1,2,3,0,0,0,NaN,NaN
4,2023,1505502,60002513,15067724,1,2,3,0,0,0,NaN,NaN


2026-07-01 02:25:25 | INFO | Amostra da tabela alunos gerada com sucesso.


## Ingestão agregada da tabela alunos

In [19]:
# Ingestão agregada da tabela alunos
# Esta visão reduz o volume e permite análises por município, ano, rede e série.

query_alunos_agregado = f"""
SELECT
  ano,
  id_municipio,
  rede,
  serie,
  COUNT(*) AS total_alunos,
  COUNTIF(proficiencia IS NOT NULL) AS total_com_proficiencia,
  AVG(proficiencia) AS media_proficiencia,
  MIN(proficiencia) AS menor_proficiencia,
  MAX(proficiencia) AS maior_proficiencia,
  COUNTIF(alfabetizado IS NOT NULL) AS total_com_status_alfabetizacao,
  COUNT(DISTINCT alfabetizado) AS qtd_status_alfabetizacao,
  COUNTIF(presenca IS NOT NULL) AS total_com_status_presenca,
  AVG(peso_aluno) AS media_peso_aluno
FROM
  `{TABLES["alunos"]}`
GROUP BY
  ano,
  id_municipio,
  rede,
  serie
ORDER BY
  ano,
  id_municipio,
  rede,
  serie
"""

alunos_agregado_file = BATCH_OUTPUT_PATH / "alunos_agregado.csv"

df_alunos_agregado = run_query_to_csv(
    query=query_alunos_agregado,
    output_file=alunos_agregado_file,
    description="Ingestão agregada da tabela alunos por ano, município, rede e série"
)

display(df_alunos_agregado.head())

write_log("Visão agregada da tabela alunos gerada com sucesso.")

2026-07-01 02:25:25 | INFO | Iniciando extração: Ingestão agregada da tabela alunos por ano, município, rede e série
2026-07-01 02:25:28 | INFO | Extração concluída: Ingestão agregada da tabela alunos por ano, município, rede e série | linhas=12431 | colunas=13 | arquivo=alunos_agregado.csv | tamanho_mb=1.0165


,ano,id_municipio,rede,serie,total_alunos,total_com_proficiencia,media_proficiencia,menor_proficiencia,maior_proficiencia,total_com_status_alfabetizacao,qtd_status_alfabetizacao,total_com_status_presenca,media_peso_aluno
0,2023,1100015,3,2,254,227,759.937724,640.637477,846.516484,254,2,254,1.118943
1,2023,1100023,3,2,1361,1222,757.466012,602.616267,853.508475,1361,2,1361,1.114386
2,2023,1100031,3,2,84,76,767.675528,650.033845,848.987394,84,2,84,1.105263
3,2023,1100049,2,2,227,200,762.040941,617.300913,846.516484,227,2,227,1.135000
4,2023,1100049,3,2,726,613,756.053475,623.151566,848.987394,726,2,726,1.185301


2026-07-01 02:25:28 | INFO | Visão agregada da tabela alunos gerada com sucesso.


## Gerar manifesto da ingestão Batch

In [20]:
# Geração do manifesto da ingestão Batch

manifest_file = BATCH_OUTPUT_PATH / "batch_ingestion_manifest.json"

with open(manifest_file, "w", encoding="utf-8") as file:
    json.dump(batch_manifest, file, ensure_ascii=False, indent=4)

write_log(f"Manifesto da ingestão Batch gerado: {manifest_file}")

df_manifest = pd.DataFrame(batch_manifest)
display(df_manifest)

2026-07-01 02:25:28 | INFO | Manifesto da ingestão Batch gerado: /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/batch_ingestion_manifest.json


,description,output_file,rows,columns,bytes_processed,file_size_mb,started_at,finished_at,elapsed_seconds,status
0,Ingestão integral da tabela meta_alfabetizacao...,/content/techchallenge-fase2-pipeline-alfabeti...,3,11,0,0.0004,2026-07-01 02:25:03,2026-07-01 02:25:05,1.19,success
1,Ingestão integral da tabela meta_alfabetizacao_uf,/content/techchallenge-fase2-pipeline-alfabeti...,81,12,0,0.0050,2026-07-01 02:25:05,2026-07-01 02:25:06,1.00,success
2,Ingestão integral da tabela meta_alfabetizacao...,/content/techchallenge-fase2-pipeline-alfabeti...,10704,13,0,0.7712,2026-07-01 02:25:06,2026-07-01 02:25:08,2.18,success
3,Ingestão integral da tabela municipio,/content/techchallenge-fase2-pipeline-alfabeti...,23995,15,0,1.3642,2026-07-01 02:25:08,2026-07-01 02:25:12,4.64,success
4,Ingestão integral da tabela uf,/content/techchallenge-fase2-pipeline-alfabeti...,145,15,0,0.0079,2026-07-01 02:25:12,2026-07-01 02:25:13,1.14,success
5,Ingestão integral da tabela dicionario,/content/techchallenge-fase2-pipeline-alfabeti...,27,5,0,0.0010,2026-07-01 02:25:13,2026-07-01 02:25:14,0.88,success
6,Ingestão controlada da tabela alunos com LIMIT...,/content/techchallenge-fase2-pipeline-alfabeti...,100000,12,0,5.3711,2026-07-01 02:25:14,2026-07-01 02:25:25,10.98,success
7,"Ingestão agregada da tabela alunos por ano, mu...",/content/techchallenge-fase2-pipeline-alfabeti...,12431,13,0,1.0165,2026-07-01 02:25:25,2026-07-01 02:25:28,3.00,success


## Validação dos arquivos gerados na Bronze

In [21]:
# Validação dos arquivos gerados na camada Bronze Batch

expected_bronze_files = [
    "meta_alfabetizacao_brasil.csv",
    "meta_alfabetizacao_uf.csv",
    "meta_alfabetizacao_municipio.csv",
    "municipio.csv",
    "uf.csv",
    "dicionario.csv",
    "alunos_sample.csv",
    "alunos_agregado.csv",
    "batch_ingestion_manifest.json",
]

bronze_validation = []

for file_name in expected_bronze_files:
    file_path = BATCH_OUTPUT_PATH / file_name

    bronze_validation.append({
        "file_name": file_name,
        "exists": file_path.exists(),
        "size_mb": round(file_path.stat().st_size / 1024 / 1024, 4) if file_path.exists() else 0
    })

df_bronze_validation = pd.DataFrame(bronze_validation)

display(df_bronze_validation)

if df_bronze_validation["exists"].all():
    write_log("Validação da camada Bronze Batch aprovada: todos os arquivos esperados foram gerados.")
else:
    missing = df_bronze_validation[df_bronze_validation["exists"] == False]["file_name"].tolist()
    write_log(f"Arquivos ausentes na Bronze Batch: {missing}", level="ERROR")

,file_name,exists,size_mb
0,meta_alfabetizacao_brasil.csv,True,0.0004
1,meta_alfabetizacao_uf.csv,True,0.0050
2,meta_alfabetizacao_municipio.csv,True,0.7712
3,municipio.csv,True,1.3642
4,uf.csv,True,0.0079
5,dicionario.csv,True,0.0010
6,alunos_sample.csv,True,5.3711
7,alunos_agregado.csv,True,1.0165
8,batch_ingestion_manifest.json,True,0.0035


2026-07-01 02:25:28 | INFO | Validação da camada Bronze Batch aprovada: todos os arquivos esperados foram gerados.


## Resumo da ingestão Batch

In [22]:
# Resumo executivo da ingestão Batch

total_files = len(expected_bronze_files)
total_rows = int(df_manifest["rows"].sum())
total_size_mb = round(df_bronze_validation["size_mb"].sum(), 4)
total_bytes_processed = int(df_manifest["bytes_processed"].sum())

summary_batch = {
    "total_files_generated": total_files,
    "total_rows_extracted": total_rows,
    "total_size_mb": total_size_mb,
    "total_bytes_processed": total_bytes_processed,
    "status": "approved" if df_bronze_validation["exists"].all() else "failed"
}

display(pd.DataFrame([summary_batch]))

write_log(
    f"Resumo Batch: arquivos={total_files}, linhas={total_rows}, "
    f"tamanho_mb={total_size_mb}, bytes_processados={total_bytes_processed}, "
    f"status={summary_batch['status']}"
)

,total_files_generated,total_rows_extracted,total_size_mb,total_bytes_processed,status
0,9,147386,8.5408,0,approved


2026-07-01 02:25:29 | INFO | Resumo Batch: arquivos=9, linhas=147386, tamanho_mb=8.5408, bytes_processados=0, status=approved


## Listar arquivos gerados

In [23]:
# Listagem dos arquivos gerados no ambiente Colab

!find /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch -maxdepth 3 -type f

/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/batch_ingestion_manifest.json
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/uf.csv
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/metadata/dataset_columns.csv
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/metadata/table_validation.json
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/metadata/dataset_tables.csv
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/metadata/dataset_storage.csv
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/meta_alfabetizacao_uf.csv
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/meta_alfabetizacao_municipio.csv
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/alunos_agregado.csv
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/alunos_sample.csv
/content/techchallenge-fase2-pipeline-alfabetizac